# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Emmamems18/flyrank-ml-internship-my-submission-assignment1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
import os
from google.colab import userdata

# Retrieve token securely from Colab Secrets
hf_token = userdata.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

print("HF_TOKEN successfully loaded into environment!")

In [1]:
import os
import duckdb
from google.colab import userdata

# 1. Load token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

# 2. Initialize DuckDB and set up Hugging Face authentication secret
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

# 3. Path to warehouse tables
rel = "hf://datasets/FlyRank/internship-warehouse"

# Quick smoke test to verify access
test_query = f"""
SELECT COUNT(*) as row_count
FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
"""

print("Running access test...")
result = con.sql(test_query).df()
print("Success! Access verified. Total dataset rows:")
print(result)

Running access test...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Success! Access verified. Total dataset rows:
   row_count
0   78835655


In [3]:
# Check exact schema of fact_daily
con.sql(f"""
DESCRIBE SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet') LIMIT 1
""").show()

┌────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│    column_name     │ column_type │  null   │   key   │ default │  extra  │
│      varchar       │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ report_date        │ DATE        │ YES     │ NULL    │ NULL    │ NULL    │
│ client_hash_id     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ content_hash_id    │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_gsc     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ client_has_ga4     │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ ga4_data_available │ BOOLEAN     │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_impressions    │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ gsc_clicks         │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

Unit of Analysis & Time Window:

Unit of Analysis: One row = One unique content page (content_hash_id) evaluated over a single calendar month.

Time Window: Mid-panel month of March 2026 (report_date in March 2026 or month = '2026-03').

Striking Distance Filter: Pages where monthly average ranking position is between 11 and 30 (gsc_sum_position / gsc_impressions BETWEEN 11 AND 30).

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Path to daily fact table
fact_daily = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"

grain_query = f"""
WITH monthly_aggregated AS (
    SELECT
        content_hash_id,
        SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) as avg_position,
        SUM(gsc_impressions) as total_impressions,
        SUM(gsc_clicks) as total_clicks,
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM {fact_daily}
    WHERE month = '2026-03'
    GROUP BY content_hash_id
    HAVING (SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0)) BETWEEN 11 AND 30
       AND SUM(gsc_impressions) >= 10
)
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT content_hash_id) as unique_content_ids,
    MIN(start_date) as earliest_date,
    MAX(end_date) as latest_date
FROM monthly_aggregated;
"""

df_grain = con.sql(grain_query).df()
print("Verification Output for March 2026 Grain:")
print(df_grain)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Verification Output for March 2026 Grain:
   total_rows  unique_content_ids earliest_date latest_date
0       37137               37137    2026-03-01  2026-03-31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Markdown Cell (Section 2)
Paste this into the Markdown cell for Section 2:

Field Sorting (4 Buckets):

Context (Identifiers & Metadata):

content_hash_id: Unique page identifier used as the primary key.

client_hash_id: Organization identifier for grouping and client-level slicing.

month: Temporal partition anchor (2026-03).

Features (Predictors knowable in March 2026):

avg_position: Current ranking proximity to Page 1 (gsc_sum_position / gsc_impressions).

gsc_impressions: Monthly organic search demand/visibility.

gsc_clicks: Historical organic traffic driven to the page.

ctr: Click-through rate (gsc_clicks / gsc_impressions).

sessions_ai: Aggregated referral traffic coming from AI search engines (ChatGPT, Gemini, Claude, Perplexity).

Label (Proxy Target):

target_priority_class: Binary target (1 = High Priority, 0 = Standard) flagged using the top 20% of the calculated opportunity_score within March 2026.

Excluded (and Why):

gsc_sum_position: Why: Uninterpretable raw total; heavily biased by impression volume. Replaced by avg_position.

Individual ai_* columns (ai_chatgpt, ai_claude, ai_gemini): Why: Too sparse and noisy individually; combined into single sessions_ai feature.

Future month data (month >= '2026-04'): Why: Violates the knowability principle and introduces direct data leakage into March decision-making.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Query to select and verify the field structure for Section 2
fields_query = f"""
SELECT
    -- Context
    content_hash_id,
    client_hash_id,
    month,

    -- Feature Inputs
    gsc_impressions,
    gsc_clicks,
    (gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as ctr,
    (gsc_sum_position * 1.0 / NULLIF(gsc_impressions, 0)) as avg_position,
    sessions_ai
FROM {fact_daily}
WHERE month = '2026-03'
LIMIT 5;
"""

df_fields = con.sql(fields_query).df()
print("Sample Extract of Categorized Fields (March 2026):")
print(df_fields)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sample Extract of Categorized Fields (March 2026):
            content_hash_id           client_hash_id    month  \
0  content_b7e512995f79d5a6  client_73cda7b4e4f265ea  2026-03   
1  content_05597932fe4da067  client_73cda7b4e4f265ea  2026-03   
2  content_7a105f548d9c6916  client_73cda7b4e4f265ea  2026-03   
3  content_905aa32a0230694e  client_73cda7b4e4f265ea  2026-03   
4  content_a3ea9792f793ec72  client_73cda7b4e4f265ea  2026-03   

   gsc_impressions  gsc_clicks    ctr  avg_position  sessions_ai  
0               20           0  0.000      3.350000         <NA>  
1                1           0  0.000      0.000000         <NA>  
2              125           1  0.008      4.928000         <NA>  
3                7           0  0.000      4.000000         <NA>  
4               11           0  0.000      2.272727         <NA>  


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Contract Claims Verification (3 Queries):

Fact 1 (Grain & Date Span): Exactly 1 row exists per unique content_hash_id in March 2026, with dates spanning strictly from 2026-03-01 to 2026-03-31.

Fact 2 (Lane Slice Volume): The striking distance slice (avg_position BETWEEN 11 AND 30) yields an exact count of qualified candidate pages for the mid-panel month.

Fact 3 (Data Availability): Filtering by gsc_data_available IS TRUE proves data health and measures how many clean, usable rows survive for feature engineering.

In [6]:
# 1. Ensure table path is explicitly set in this cell
fact_daily = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"

# -------------------------------------------------------------
# Query 1: Prove Grain & Date Span
# -------------------------------------------------------------
print("Running Query 1: Grain & Date Span...")
q1_grain = f"""
SELECT
    COUNT(DISTINCT content_hash_id) as total_unique_pages,
    MIN(report_date) as earliest_date,
    MAX(report_date) as latest_date
FROM {fact_daily}
WHERE month = '2026-03';
"""
df_q1 = con.sql(q1_grain).df()
print("Query 1 Results:")
print(df_q1)
print("-" * 50)

# -------------------------------------------------------------
# Query 2: Prove Lane Slice Row Count (Positions 11-30)
# -------------------------------------------------------------
print("Running Query 2: Striking Distance Page Count...")
q2_counts = f"""
WITH page_positions AS (
    SELECT
        content_hash_id,
        SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as avg_pos,
        SUM(gsc_impressions) as total_imp
    FROM {fact_daily}
    WHERE month = '2026-03'
    GROUP BY content_hash_id
)
SELECT
    COUNT(*) as striking_distance_pages
FROM page_positions
WHERE avg_pos BETWEEN 11 AND 30
  AND total_imp >= 10;
"""
df_q2 = con.sql(q2_counts).df()
print("Query 2 Results:")
print(df_q2)
print("-" * 50)

# -------------------------------------------------------------
# Query 3: Availability Verification (Filtering with IS TRUE)
# -------------------------------------------------------------
print("Running Query 3: Availability Check (IS TRUE)...")
q3_availability = f"""
SELECT
    COUNT(*) as total_daily_records,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) as gsc_true_count,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) as ga4_true_count
FROM {fact_daily}
WHERE month = '2026-03';
"""
df_q3 = con.sql(q3_availability).df()
print("Query 3 Results:")
print(df_q3)

Running Query 1: Grain & Date Span...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 1 Results:
   total_unique_pages earliest_date latest_date
0              331437    2026-03-01  2026-03-31
--------------------------------------------------
Running Query 2: Striking Distance Page Count...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Query 2 Results:
   striking_distance_pages
0                    37137
--------------------------------------------------
Running Query 3: Availability Check (IS TRUE)...
Query 3 Results:
   total_daily_records  gsc_true_count  ga4_true_count
0              9841378         3611061          413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Data Limits & Blindspots (What This Data Can Never Tell You):

Asymmetric Telemetry (GSC vs. GA4 Gap): Search Console tracking (gsc_data_available IS TRUE) covers approximately 3.61 million daily records, whereas GA4 behavioral data (ga4_data_available IS TRUE) covers only 413,966 records (~11.5%). The dataset cannot provide on-page engagement signals (bounce rate, scroll depth, dwell time) for nearly 90% of pages.

Unbalanced Client History: Clients onboarded at different dates throughout the warehouse timeline. Newer clients lack deep multi-month historical trends, creating asymmetric feature density across rows.

Off-Page & SERP Blindspots: The dataset tracks internal performance outcomes (impressions, position, clicks) but cannot capture external SERP dynamics—such as Google core algorithm updates, competitor backlink additions, or search intent shifts.

Aggregated Window Blur: Converting 31 daily snapshot rows into a single monthly average (avg_position) masks intra-month volatility, such as a page dropping from position 11 to 29 halfway through the month due to a penalty or site redesign.

In [7]:
# Prove the data coverage imbalance between GSC and GA4
limits_query = f"""
SELECT
    COUNT(*) as total_daily_records,
    COUNT(CASE WHEN gsc_data_available IS TRUE THEN 1 END) as gsc_records,
    COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) as ga4_records,
    ROUND(100.0 * COUNT(CASE WHEN ga4_data_available IS TRUE THEN 1 END) / COUNT(*), 2) as ga4_coverage_pct,
    COUNT(CASE WHEN gsc_data_available IS TRUE AND ga4_data_available IS TRUE THEN 1 END) as combined_telemetry_records
FROM {fact_daily}
WHERE month = '2026-03';
"""

df_limits = con.sql(limits_query).df()

print("Data Limits Proof (Telemetry Coverage Gap in March 2026):")
print(df_limits)
print(f"\nLimitation Confirmed: Only {df_limits['ga4_coverage_pct'].iloc[0]}% of daily rows contain GA4 behavioral metrics.")

Data Limits Proof (Telemetry Coverage Gap in March 2026):
   total_daily_records  gsc_records  ga4_records  ga4_coverage_pct  \
0              9841378      3611061       413966              4.21   

   combined_telemetry_records  
0                      364347  

Limitation Confirmed: Only 4.21% of daily rows contain GA4 behavioral metrics.


##5. Features + The Leakage Trap

5-Feature Matrix & Knowability Justification:

avg_position: Knowable at decision moment because it is calculated strictly from GSC search impression distributions within the observation month (2026-03).

gsc_impressions: Knowable at decision moment because it aggregates historical search visibility logged prior to 2026-03-31.

gsc_clicks: Knowable at decision moment because it records historical organic traffic generated during the observation window.

ctr: Knowable at decision moment because it is derived directly from historical clicks and impressions (clicks / impressions).

sessions_ai: Knowable at decision moment because it sums historical referral traffic from AI engines logged up to the final day of the month.

In [8]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score

# 1. Build honest feature matrix + label for March 2026
feature_query = f"""
WITH monthly_features AS (
    SELECT
        content_hash_id,
        SUM(gsc_impressions) as gsc_impressions,
        SUM(gsc_clicks) as gsc_clicks,
        SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as ctr,
        SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0) as avg_position,
        COALESCE(SUM(sessions_ai), 0) as sessions_ai
    FROM {fact_daily}
    WHERE month = '2026-03'
    GROUP BY content_hash_id
    HAVING (SUM(gsc_sum_position) * 1.0 / NULLIF(SUM(gsc_impressions), 0)) BETWEEN 11 AND 30
       AND SUM(gsc_impressions) >= 10
)
SELECT *,
       -- Define proxy opportunity score
       (gsc_impressions * (31 - avg_position)) as opportunity_score
FROM monthly_features;
"""

df_features = con.sql(feature_query).df()

# 2. Create target priority class (Top 20% = High Priority Class 1)
threshold = df_features["opportunity_score"].quantile(0.80)
df_features["target_priority_class"] = (df_features["opportunity_score"] >= threshold).astype(int)

# 3. Define Honest Features
honest_cols = ["gsc_impressions", "gsc_clicks", "ctr", "avg_position", "sessions_ai"]
X_honest = df_features[honest_cols].fillna(0)
y = df_features["target_priority_class"]

# 4. INTRODUCE THE LEAKAGE TRAP (Deliberately add label-derived column)
# Adding direct proxy target component causes data leakage
X_leaked = X_honest.copy()
X_leaked["LEAKED_opportunity_score"] = df_features["opportunity_score"]

# Evaluate Model WITH Leakage
clf_leaked = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_leaked.fit(X_leaked, y)
pred_leaked = clf_leaked.predict(X_leaked)
score_leaked = precision_score(y, pred_leaked)

# Evaluate Model WITHOUT Leakage (Honest Score)
clf_honest = DecisionTreeClassifier(max_depth=3, random_state=42)
clf_honest.fit(X_honest, y)
pred_honest = clf_honest.predict(X_honest)
score_honest = precision_score(y, pred_honest)

print("=== THE LEAKAGE TRAP EXPERIMENT ===")
print(f"Precision WITH Leaked Feature : {score_leaked * 100:.2f}% (Artificially Perfect Trap)")
print(f"Precision WITHOUT Leaked Feature: {score_honest * 100:.2f}% (Honest Baseline Score)")
print("\nLeakage Trap Removed. Feature Matrix is clean and ready!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== THE LEAKAGE TRAP EXPERIMENT ===
Precision WITH Leaked Feature : 100.00% (Artificially Perfect Trap)
Precision WITHOUT Leaked Feature: 88.79% (Honest Baseline Score)

Leakage Trap Removed. Feature Matrix is clean and ready!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.